# Supply Chain Late Delivery Prediction - Feature Engineering & Feature Selection
**Student:** IT24104225 - Weerathunga W.M.C.N.  
**Assigned Responsibility:** Feature Engineering & Feature Selection  
**Target Variable:** `Late_delivery_risk` (1 = Late delivery, 0 = On-time / Advance)  

---

## 1. Project Setup & Initial Dataset Inspection
I first loaded the raw DataCo Supply Chain dataset and checked its structure, column types, missing values, and the target distribution before designing any feature engineering or selection steps.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_selection import mutual_info_classif
import warnings
warnings.filterwarnings('ignore')

# Set display options for clean inspection
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 1000)

DATA_PATH = "data/DataCoSupplyChainDataset.csv"
df = pd.read_csv(DATA_PATH, encoding="latin1")

print("Dataset loaded successfully.")
print("Dataset Shape:", df.shape)


Dataset loaded successfully.
Dataset Shape: (180519, 53)



In [2]:
# Inspect target variable, missing values, and duplicate rows
TARGET = "Late_delivery_risk"

print("Target column:", TARGET)
print("\nTarget distribution:")
print(df[TARGET].value_counts())

print("\nTarget proportions:")
print(df[TARGET].value_counts(normalize=True))

print("\nMissing values summary (columns with missing values):")
missing = df.isnull().sum()
print(missing[missing > 0])

print("\nDuplicate full rows count:", df.duplicated().sum())
print("Total unique Order Ids:", df["Order Id"].nunique())


Target column: Late_delivery_risk

Target distribution:
Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

Target proportions:
Late_delivery_risk
1    0.548291
0    0.451709
Name: proportion, dtype: float64

Missing values summary (columns with missing values):
Customer Lname              8
Customer Zipcode            3
Order Zipcode          155679
Product Description    180519
dtype: int64

Duplicate full rows count: 0
Total unique Order Ids: 65752



## 2. Grouped Train / Test Split
I defined `Late_delivery_risk` as the target variable. Because an individual customer order can contain multiple item records, rows sharing the same `Order Id` are correlated. To prevent data leakage across splits, I used `Order Id` as the grouping variable with an 80/20 grouped split using `GroupShuffleSplit`.

I then verified that no `Order Id` appears in both the training set and the testing set.


In [3]:
# Perform grouped train/test split (80% train, 20% test) grouped by Order Id
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["Order Id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

# Verify Order Id grouping and check for any overlap
train_orders = set(train_df["Order Id"])
test_orders = set(test_df["Order Id"])
overlap = train_orders.intersection(test_orders)

print(f"Unique Order Ids in Train: {len(train_orders):,}")
print(f"Unique Order Ids in Test:  {len(test_orders):,}")
print(f"Overlap in Order Ids between Train and Test: {len(overlap)}")

# Verify target balance across splits
print("\nTrain Target Proportions:")
print(train_df[TARGET].value_counts(normalize=True).round(4))

print("\nTest Target Proportions:")
print(test_df[TARGET].value_counts(normalize=True).round(4))


Train shape: (144650, 53)
Test shape: (35869, 53)
Unique Order Ids in Train: 52,601
Unique Order Ids in Test:  13,151
Overlap in Order Ids between Train and Test: 0

Train Target Proportions:
Late_delivery_risk
1    0.5483
0    0.4517
Name: proportion, dtype: float64

Test Target Proportions:
Late_delivery_risk
1    0.5484
0    0.4516
Name: proportion, dtype: float64



## 3. Leakage Feature & Column Role Investigation
Before engineering new features or making selection decisions, I checked each column in the dataset to determine if it causes post-delivery leakage or contains redundant identifier/constant information.

### Target Leakage Analysis
1. `Days for shipping (real)`: Represents the actual number of days taken for shipping. This is only known after the shipment is completed. Furthermore, comparing real shipping days to scheduled shipping days directly dictates the target label (`Late_delivery_risk`).
2. `Delivery Status`: Categorical delivery outcome (`Late delivery`, `Advance shipping`, `Shipping on time`, `Shipping canceled`). This directly leaks whether the order was delivered late.
3. `shipping date (DateOrders)`: The actual timestamp when shipment was dispatched/completed, which is a post-order event.
4. `Order Status`: Post-fulfillment transaction status (`COMPLETE`, `PENDING`, `CLOSED`, etc.) reflecting outcomes after fulfillment processing.

### Identifiers, PII, and Constant Columns
- `Customer Email`, `Customer Password`: Masked constant strings (`XXXXXXXXX`), PII.
- `Customer Fname`, `Customer Lname`: Customer names (PII, no predictive generalization).
- `Customer Street`: Granular street address (7,458 unique values), high cardinality PII.
- `Product Description`: 100% missing (NaN for all rows).
- `Product Status`: Constant value 0 across all records (zero variance).
- `Product Image`: URL string redundant with Product Name / Product Card Id.
- `Order Zipcode`: Missing in ~86.2% of records (155,679 nulls).
- `Order Item Id`: Unique transaction row index (180,519 unique values), arbitrary identifier causing overfitting.
- `Order Id`: Transaction identifier; retained strictly for grouping and verification, not as an input feature.


In [4]:
# Detailed inspection of target leakage columns
print("--- 1. Days for shipping (real) vs Scheduled days and Late delivery risk ---")
print(train_df.groupby(["Days for shipment (scheduled)", "Days for shipping (real)"])["Late_delivery_risk"].agg(["count", "mean"]).head(10))

print("\n--- 2. Delivery Status vs Late_delivery_risk ---")
print(train_df.groupby("Delivery Status")["Late_delivery_risk"].agg(["count", "mean"]))

print("\n--- 3. Constant and Masked Columns ---")
print("Customer Email unique values:", train_df["Customer Email"].unique())
print("Customer Password unique values:", train_df["Customer Password"].unique())
print("Product Status unique values:", train_df["Product Status"].unique())
print("Product Description missing percentage: {:.2f}%".format(train_df["Product Description"].isnull().mean() * 100))


--- 1. Days for shipping (real) vs Scheduled days and Late delivery risk ---
                                                        count      mean
Days for shipment (scheduled) Days for shipping (real)                 
0                             0                          3996  0.000000
                              1                          3771  0.957306
1                             2                         22229  0.953259
2                             2                          5716  0.000000
                              3                          5600  0.957500
                              4                          5673  0.974088
                              5                          5699  0.962274
                              6                          5514  0.959557
4                             2                         17515  0.000000
                              3                         17335  0.000000

--- 2. Delivery Status vs Late_delivery_risk ---
         

In [5]:
# Define explicit leakage and unusable column categories
leakage_features = [
    "Days for shipping (real)",
    "Delivery Status",
    "shipping date (DateOrders)",
    "Order Status"
]

identifier_and_unusable_features = [
    "Order Item Id",
    "Customer Email",
    "Customer Password",
    "Customer Fname",
    "Customer Lname",
    "Customer Street",
    "Product Description",
    "Product Status",
    "Product Image",
    "Order Zipcode"
]

print("Identified Leakage Features (to be excluded):", leakage_features)
print("Identified Identifier/Unusable Features (to be excluded):", identifier_and_unusable_features)


Identified Leakage Features (to be excluded): ['Days for shipping (real)', 'Delivery Status', 'shipping date (DateOrders)', 'Order Status']
Identified Identifier/Unusable Features (to be excluded): ['Order Item Id', 'Customer Email', 'Customer Password', 'Customer Fname', 'Customer Lname', 'Customer Street', 'Product Description', 'Product Status', 'Product Image', 'Order Zipcode']



## 4. Feature Engineering
In this stage, I derived new informative features using only information that would be known before shipping occurs.

### 4.1 Temporal Feature Engineering
I extracted temporal information from `order date (DateOrders)`:
- `order_year`: Year when the order was placed.
- `order_month`: Month of the order (1-12) to capture seasonal demand spikes and holiday logistics.
- `order_day`: Day of month (1-31).
- `order_day_of_week`: Day of the week (0 = Monday, 6 = Sunday) to capture weekend vs weekday dispatch patterns.
- `order_hour`: Hour of order placement (0-23) to capture daily ordering cutoffs.
- `is_weekend`: Binary flag indicating whether the order was placed on Saturday or Sunday.
- `order_quarter`: Financial quarter (1-4).

These features were derived separately for both the training and testing sets to maintain consistency.


In [6]:
# 4.1 Temporal Feature Engineering from order date (DateOrders)
train_df["order_datetime"] = pd.to_datetime(train_df["order date (DateOrders)"])
test_df["order_datetime"] = pd.to_datetime(test_df["order date (DateOrders)"])

for dataset in [train_df, test_df]:
    dataset["order_year"] = dataset["order_datetime"].dt.year
    dataset["order_month"] = dataset["order_datetime"].dt.month
    dataset["order_day"] = dataset["order_datetime"].dt.day
    dataset["order_day_of_week"] = dataset["order_datetime"].dt.dayofweek
    dataset["order_hour"] = dataset["order_datetime"].dt.hour
    dataset["is_weekend"] = dataset["order_day_of_week"].isin([5, 6]).astype(int)
    dataset["order_quarter"] = dataset["order_datetime"].dt.quarter

print("Temporal features derived successfully.")
print(train_df[["order date (DateOrders)", "order_year", "order_month", "order_day", "order_day_of_week", "order_hour", "is_weekend", "order_quarter"]].head(5))


Temporal features derived successfully.
  order date (DateOrders)  order_year  order_month  order_day  order_day_of_week  order_hour  is_weekend  order_quarter
0         1/31/2018 22:56        2018            1         31                  2          22           0              1
1         1/13/2018 12:27        2018            1         13                  5          12           1              1
4         1/13/2018 11:24        2018            1         13                  5          11           1              1
5         1/13/2018 11:03        2018            1         13                  5          11           1              1
6         1/13/2018 10:42        2018            1         13                  5          10           1              1

